<a href="https://colab.research.google.com/github/Achiever55/Brain_tumor_MRI_image/blob/main/Brain_Tumor_MRI_Image_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Detection and Classification of Brain Tumor through Brain MRI Images **  

![Brain MRI Images Cover Photo](https://media.gettyimages.com/photos/head-scan-perspective-picture-id184978066?s=2048x2048)

## **1.Introduction**  

* The occurrence of brain tumor patients in India is steadily rising,
more and more number of cases are reported each year in India across
various age groups.  

* The [International Association of Cancer Registries (IARC)](https://cutt.ly/Wc4DaIE) reported
that there are over $28,000$ cases of brain tumours reported in India
each year and more than $24,000$ people reportedly i.e. $85.72\%$ of the
total reported die due to brain tumours annually. Brain tumour’s are
a serious condition and in most cases fatal if not detected & treated
in early stages.



## **2. Setting Up Local Storage for Dataset**

### **2.1 Giving Access To Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/' ,force_remount=True)

Mounted at /content/gdrive/


### **2.2 Checking OS Version and Details**

In [ ]:
print("OS Version & Details: ")
!lsb_release -a

OS Version & Details: 
No LSB modules are available.
Distributor ID:	Ubuntu
Description:	Ubuntu 22.04.5 LTS
Release:	22.04
Codename:	jammy


## **3. Importing Required Libraries**

In [ ]:
import sys
import os
import math

import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
from matplotlib import rcParams
rcParams['figure.dpi'] = 300
%matplotlib inline
import seaborn as sns
import missingno as msno
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import *
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import *

from PIL import Image, ImageEnhance
from tensorflow.keras.preprocessing.image import *

print(f'Tensorflow Version: {tf.__version__}.')

Tensorflow Version: 2.20.0.


In [ ]:
pip install pynvml

## **4. Setting Up the Environment**

In [ ]:
gpu_device_location = tpu_device_location = cpu_device_location = None

if os.environ.get('COLAB_GPU') == '1':
    print("Allocated GPU Runtime Details:")
    !nvidia-smi
    print()
    try:
        import pynvml
        pynvml.nvmlInit()
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        gpu_device_name = pynvml.nvmlDeviceGetName(handle)

        # Ensure name is a string for comparison
        if isinstance(gpu_device_name, bytes):
            gpu_device_name = gpu_device_name.decode('utf-8')

        # Updated check using standard strings
        allowed_gpus = {'Tesla T4', 'Tesla P4', 'Tesla P100-PCIE-16GB', 'Tesla T4'}
        if gpu_device_name not in allowed_gpus:
             raise Exception(f"Unfortunately this instance has a {gpu_device_name}.\n"
                            "If you need a T4, P4 or P100, factory reset your runtime.")
    except Exception as hardware_exception:
        print(hardware_exception, end = '\n\n')

    gpu_device_location = tf.test.gpu_device_name()
    print(f"{gpu_device_name} is allocated successfully at location: {gpu_device_location}")

elif 'COLAB_TPU_ADDR' in os.environ:
    tpu_device_location = f"grpc://{os.environ['COLAB_TPU_ADDR']}"
    print(f"TPU is allocated successfully at location: {tpu_device_location}.")
    resolver = tf.distribute.cluster_resolver.TPUClusterResolver(tpu_device_location)
    tf.config.experimental_connect_to_cluster(resolver)
    tf.tpu.experimental.initialize_tpu_system(resolver)
    tpu_strategy = tf.distribute.TPUStrategy()
else:
    cpu_device_location = "/cpu:0"
    print("GPUs and TPUs are not allocated successfully, hence runtime fallbacked to CPU.")

### **4.1 Installation of `tree` Utility Using `Bash`.**  

In [ ]:
%%bash
RED_COLOR='\033[0;31m'
NO_COLOR='\033[0m'
pkg_name=tree
dpkg -s $pkg_name &> /dev/null
if [ "$?" -ne "0" ]
    then
        echo "Installing tree utility..."
        apt-get autoclean
        apt-get autoremove
        apt-get install $pkg_name
        if [ "$?" -eq "0" ]
            then
                echo -e ${RED_COLOR}"tree utility installed sucessfully.\n"${NO_COLOR}
        fi
    else
        echo "tree utility is already installed."
fi
tree --version

### **4.2 Display of File Structure**

In [ ]:
!tree -d -C "/content/gdrive/MyDrive/brain-tumor-mri-dataset"

### **4.3 Setting Up Paths to Root and Data Directories**

In [ ]:
import os

# Define the base path for Google Drive
GOOGLE_DRIVE_PATH = r"/content/gdrive/MyDrive/"
DATASET_NAME = "brain-tumor-mri-dataset" # The name of your dataset folder

# Set ROOT_DIR to the dataset's base path for consistency with subsequent cells
ROOT_DIR = os.path.join(GOOGLE_DRIVE_PATH, DATASET_NAME)

# DATA_ROOT_DIR is the same as ROOT_DIR in this context
DATA_ROOT_DIR = ROOT_DIR

# Verify the dataset root directory
if not os.path.isdir(DATA_ROOT_DIR):
    print(f"ERROR: The dataset root directory '{DATA_ROOT_DIR}' was not found.")
    print(f"This usually means the folder '{DATASET_NAME}' is not in your Google Drive's MyDrive, or its name is misspelled.")
    print("Please ensure the following:")
    print("1. You have run the cell that mounts Google Drive (cell '1zkla8QFK9B8').")
    print("2. You have granted all necessary permissions when prompted by Google Drive.")
    print(f"3. The folder '{DATASET_NAME}' exists directly in your Google Drive's 'MyDrive' and is spelled correctly (case-sensitive).")
    print("4. Sometimes, restarting the runtime (Runtime > Restart runtime) and re-running all cells can resolve transient issues.")
    raise FileNotFoundError(f"Dataset root directory not found: {DATA_ROOT_DIR}. Please check Google Drive path and permissions.")


# Note: Ensure 'Training' and 'Tumor-Mask' exist inside 'brain-tumor-mri-dataset'
TRAIN_DIR = os.path.join(DATA_ROOT_DIR, 'Training')
MASK_DIR = os.path.join(DATA_ROOT_DIR, 'Tumor-Mask')

# If these next two fail, check if 'Training' and 'Tumor-Mask' are capitalized correctly in Drive
if os.path.isdir(TRAIN_DIR) and os.path.isdir(MASK_DIR):
    TUMOR_CLASS = ['meningioma', 'glioma', 'pituitary', 'notumor']
    IMAGE_DATA_PATHS = [os.path.join(TRAIN_DIR, tumor_class) for tumor_class in TUMOR_CLASS]
    MASK_DATA_PATHS = [os.path.join(MASK_DIR, tumor_name) for tumor_name in TUMOR_CLASS[:-1]]
    print("Paths set up and verified successfully!")
else:
    print(f"Warning: Subfolders 'Training' or 'Tumor-Mask' are missing within '{DATA_ROOT_DIR}'.")
    print("Please verify their existence and capitalization in your Google Drive.")

In [ ]:
# Install gdown to download folders from Google Drive
!pip install gdown

# Define the Google Drive folder ID from the provided link
GOOGLE_DRIVE_FOLDER_ID = 'https://drive.google.com/drive/folders/1A55WOtPcfpOB5EOzIMK1KkwaZEj3-ueU?usp=sharing'
# Define the local directory where the dataset will be downloaded
LOCAL_DATASET_PATH = '/content/brain-tumor-mri-dataset'

print(f"Downloading dataset from Google Drive Folder ID: {GOOGLE_DRIVE_FOLDER_ID} to {LOCAL_DATASET_PATH}...")

# Use gdown to download the folder
# Note: This requires the Google Drive folder to be shared as 'Anyone with the link can view'
import gdown
gdown.download_folder(id=GOOGLE_DRIVE_FOLDER_ID, output=LOCAL_DATASET_PATH, quiet=False)

print(f"Dataset downloaded to {LOCAL_DATASET_PATH}.")

# Verify the downloaded folder
if not os.path.isdir(LOCAL_DATASET_PATH):
    raise FileNotFoundError(f"Failed to download dataset to {LOCAL_DATASET_PATH}. Please ensure the folder is publicly shared via link.")

In [ ]:
pip install -U PyDrive2

Now, we need to authenticate with Google Drive. This will open a new tab for you to grant permissions. Please follow the prompts to complete the authentication.

In [ ]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials # Import to bridge Colab auth with PyDrive2

# 1. Authenticate with Google Colab
auth.authenticate_user()

# 2. Create GoogleAuth instance with credentials from Colab authentication
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default() # Link Colab's credentials

# 3. Create GoogleDrive instance with authenticated GoogleAuth
pydrive_drive = GoogleDrive(gauth) # Renamed to avoid conflict

print("Authentication successful!")

Now that we are authenticated, we can list the files and folders within the `brain-tumor-mri-dataset` folder using its ID. We will list the top-level contents of this folder.

In [ ]:
# The GOOGLE_DRIVE_FOLDER_ID is currently a URL. We need to extract the ID from it.
# Example URL: 'https://drive.google.com/drive/folders/1A55WOtPcfpOB5EOzIMK1KkwaZEj3-ueU?usp=sharing'

# Extracting the folder ID from the URL
import re
match = re.search(r'id=([\w-]+)|folders/([\w-]+)', GOOGLE_DRIVE_FOLDER_ID)
if match:
    dataset_folder_id = match.group(1) or match.group(2)
else:
    dataset_folder_id = GOOGLE_DRIVE_FOLDER_ID # Assume it's just the ID if regex fails

print(f"Extracted Google Drive Folder ID: {dataset_folder_id}")

# List files and folders in the specified directory using PyDrive2's pydrive_drive object
file_list = pydrive_drive.ListFile({'q': f"'{dataset_folder_id}' in parents and trashed=false"}).GetList()

print(f"Found {len(file_list)} items in the folder:\n")
for file in file_list:
    print(f"Title: {file['title']}, ID: {file['id']}, Type: {file['mimeType']}")

In [ ]:
import os

# Ensure the local base directory for the dataset exists
os.makedirs(LOCAL_DATASET_PATH, exist_ok=True)
print(f"Created local dataset directory: {LOCAL_DATASET_PATH}")

# Iterate through the top-level folders (meningioma, notumor, glioma, pituitary) obtained from file_list
for folder_item in file_list:
    if folder_item['mimeType'] == 'application/vnd.google-apps.folder':
        folder_title = folder_item['title']
        folder_id = folder_item['id']
        local_subfolder_path = os.path.join(LOCAL_DATASET_PATH, folder_title)
        os.makedirs(local_subfolder_path, exist_ok=True)
        print(f"\nDownloading contents of folder '{folder_title}' (ID: {folder_id}) to {local_subfolder_path}...")

        # List files within this subfolder using PyDrive2
        subfolder_files = pydrive_drive.ListFile({'q': f"'{folder_id}' in parents and trashed=false"}).GetList()

        for file_in_subfolder in subfolder_files:
            # PyDrive2's ListFile can return files or nested folders. Download only files.
            if file_in_subfolder['mimeType'] != 'application/vnd.google-apps.folder':
                file_title = file_in_subfolder['title']
                file_id = file_in_subfolder['id']
                local_file_path = os.path.join(local_subfolder_path, file_title)
                print(f"  Downloading file '{file_title}' (ID: {file_id})")
                file_in_subfolder.GetContentFile(local_file_path)
            else:
                print(f"  Skipping nested subfolder: {file_in_subfolder['title']}") # This script assumes a flat structure within main classes.

print("\nDataset download complete to local path!")

# After downloading, verify the structure with tree
print(f"\nVerifying downloaded dataset structure under {LOCAL_DATASET_PATH}:")
!tree "$LOCAL_DATASET_PATH"

The dataset has been downloaded to a local path. Now, we will set the `ROOT_DIR` and other paths to this downloaded dataset location.

In [ ]:
import os

# Re-define ROOT_DIR and DATA_ROOT_DIR as they were not defined in this execution context
# These definitions are copied from cell zkxWgEs-CXgi
ROOT_DIR = r"/content/gdrive/MyDrive/"
ROOT_DIR = os.path.join(ROOT_DIR, "brain-tumor-mri-dataset")
DATA_ROOT_DIR = os.path.join(ROOT_DIR, "brain-tumor-mri-dataset")

print(f"Listing contents of: {os.path.dirname(ROOT_DIR)}")
!ls -F "{os.path.dirname(ROOT_DIR)}"

print(f"\nAttempting to list contents of the problematic directory: {ROOT_DIR}")
!ls -F "{ROOT_DIR}"


## **5. Data Preprocessing and Exploratory Data Analysis**

In [ ]:
import os

# 1. Set the correct Data Root
ROOT_DIR = r"/content/gdrive/MyDrive/"
DATA_ROOT_DIR = os.path.join(ROOT_DIR, "brain-tumor-mri-dataset")

# 2. Update classes to match your 'tree' output exactly (lowercase, no underscores)
TUMOR_CLASS = ['meningioma', 'glioma', 'pituitary', 'notumor']

# 3. Define paths directly (since 'Training' folder doesn't exist in your structure)
IMAGE_DATA_PATHS = [os.path.join(DATA_ROOT_DIR, tumor_class) for tumor_class in TUMOR_CLASS]

# 4. Verification
for path in IMAGE_DATA_PATHS:
    if not os.path.isdir(path):
        print(f"Warning: Folder not found: {path}")
    else:
        print(f"Verified: {path}")

print("\nVariables IMAGE_DATA_PATHS and TUMOR_CLASS are now defined.")

### **5.1 Data Distribution Visualization**

In [ ]:
# Calculate the number of images in each folder
data_distribution_count = pd.Series(
    [len(os.listdir(path)) for path in IMAGE_DATA_PATHS],
    index=TUMOR_CLASS
)

# Display the counts to verify
print(data_distribution_count)

### **5.2 Visualisation of Brain MRI Dataset**

**Dataset Source: https://figshare.com/articles/dataset/brain_tumor_dataset/1512427**  

**Source Code for Conversion of `.mat` file to `.jpg`: [Google Colab Notebook Link](https://colab.research.google.com/drive/1aucu3Ipj1eS0y1YEzKq76Z38TaetHSc3?usp=sharing)**  

**Final Dataset Link: https://drive.google.com/drive/folders/11QIC82FBdAyq0PUwLVNd22i-oq6lcat1?usp=sharing**

In [ ]:
  BRIGHTNESS_FACTOR = 1.7

  # 1. Define MASK_DATA_PATHS (since 'Tumor-Mask' folder was missing in your tree)
  # If you don't have masks, we will just plot the first 4 images from the dataset
  fig, axes = plt.subplots(nrows = 2, ncols = 2, figsize = (12, 10))
  axes = axes.flatten()

  # 2. Fix the TypeError by removing fontdict and using fontweight directly
  fig.suptitle("Brain Tumor MRI Images (T2w)", fontsize = 16, fontweight = 'bold', y = 1.02)

  for curr_title, filename, curr_axis in zip(TUMOR_CLASS, IMAGE_DATA_PATHS, axes):
      # Get the 3rd image from the folder
      img_list = os.listdir(filename)
      curr_image = Image.open(os.path.join(filename, img_list[2]))

      # Enhance and Plot
      img_enhancer = ImageEnhance.Brightness(curr_image)
      curr_axis.imshow(img_enhancer.enhance(BRIGHTNESS_FACTOR), cmap='gray')

      # Clean up title formatting
      clean_title = curr_title.replace('_', ' ').title()
      curr_axis.set_title(clean_title, fontsize = 14)
      curr_axis.axis('off')

  fig.tight_layout()
  sns.despine(left=True, bottom=True)
  plt.show()

### **6. Development of Training, Validation & Testing Dataset**

In [ ]:
image_data_paths = []
for curr_path, tumor_name in zip(IMAGE_DATA_PATHS, TUMOR_CLASS):
    if os.path.exists(curr_path) and os.path.isdir(curr_path):
        image_data_paths.extend(map(lambda filename: (os.path.join(curr_path, filename), tumor_name), os.listdir(curr_path)))

In [ ]:
image_data_paths_df = pd.DataFrame(image_data_paths, columns = ['image_filepaths', 'tumor_class']).sample(frac = 1, random_state = 42).reset_index(drop = True)
image_data_paths_df.head()

In [ ]:
image_data_paths_df.info()

In [ ]:
intermediate_train_data, test_data = train_test_split(image_data_paths_df,
                                                      train_size = 0.70,
                                                      random_state = 42,
                                                      stratify = image_data_paths_df.tumor_class)

train_data, validation_data = train_test_split(intermediate_train_data,
                                               train_size = 0.80,
                                               random_state = 42,
                                               stratify = intermediate_train_data.tumor_class)

### **6.1 Training, Validation and Testing Dataset Data Distribution Visualization**

In [ ]:
fig, axes = plt.subplots(ncols = 3, figsize = (20, 5))

# FIX: Removed fontdict to avoid the duplicate 'weight'/'fontweight' error
fig.suptitle("Distribution of Training/Validation/Testing Data",
             fontsize = 16, fontweight = 'bold', y = 1.05)

# Plotting the three distributions
sns.countplot(x = train_data.tumor_class, order = TUMOR_CLASS, ax = axes[0])
sns.countplot(x = validation_data.tumor_class, order = TUMOR_CLASS, ax = axes[1])
sns.countplot(x = test_data.tumor_class, order = TUMOR_CLASS, ax = axes[2])

for curr_axis, curr_title in zip(axes, ['Train Data', 'Validation Data', 'Test Data']):
    curr_axis.grid(False, alpha = 0.1)
    curr_axis.set_title(curr_title, fontsize = 12)
    curr_axis.set_xlabel("Tumor Classes", fontsize = 12)
    curr_axis.set_ylabel("Total Observations", fontsize = 12)
    curr_axis.tick_params(which = 'major', labelsize = 10)

    # Updated label logic: handles names like 'notumor' or 'meningioma'
    labels = [xtick.replace('_', '\n').title() for xtick in TUMOR_CLASS]
    curr_axis.set_xticklabels(labels)

sns.despine()
plt.show()

## **7. Data/Image Augmentation**
* Image augmentation is usually used to increase the image dataset and also to make the network more robust against translation invariance. Image augmentation is defined as creating duplicates of the original image datasets by flipping, rotating, zooming, and adjusting brightness.

* We will use data/image augmentation using `ImageDataGenerator` class to train the model on different types of combinations formed by rotation, flipping, changing the brightness etc of an image so as to increase our model accuracy.

In [ ]:
image_size = 128
batch_size = 32

image_datagen_kwargs = dict(rescale = 1 / 255,
                            rotation_range = 15,
                            width_shift_range = 0.1,
                            zoom_range = 0.01,
                            shear_range = 0.01,
                            brightness_range = [0.3, 1.5],
                            horizontal_flip = True,
                            vertical_flip = True)

In [ ]:
train_image_datagen = ImageDataGenerator(**image_datagen_kwargs)
validation_image_datagen = ImageDataGenerator(**image_datagen_kwargs)
test_image_datagen = ImageDataGenerator(**image_datagen_kwargs)

In [ ]:
train_dataset = train_image_datagen.flow_from_dataframe(train_data,
                                                        x_col = 'image_filepaths',
                                                        y_col = 'tumor_class',
                                                        seed = 42,
                                                        batch_size = batch_size,
                                                        target_size = (image_size, image_size),
                                                        color_mode = 'rgb')
validation_dataset = validation_image_datagen.flow_from_dataframe(validation_data,
                                                                  x_col = 'image_filepaths',
                                                                  y_col = 'tumor_class',
                                                                  seed = 42,
                                                                  batch_size = batch_size,
                                                                  target_size = (image_size, image_size),
                                                                  color_mode = 'rgb')
test_dataset = test_image_datagen.flow_from_dataframe(test_data,
                                                      x_col = 'image_filepaths',
                                                      y_col = 'tumor_class',
                                                      seed = 42,
                                                      batch_size = batch_size,
                                                      target_size = (image_size, image_size),
                                                      color_mode = 'rgb')

In [ ]:
print("Information about Training Dataset:")
print(train_dataset.class_indices)
print(train_dataset.image_shape, end = '\n\n')

print("Information about Validation Dataset:")
print(validation_dataset.class_indices)
print(validation_dataset.image_shape, end = '\n\n')

print("Information about Testing Dataset:")
print(test_dataset.class_indices)
print(test_dataset.image_shape)

### **7.1 Training Data Images Glimpse**

In [ ]:
fig, axes = plt.subplots(nrows = 2, ncols = 8, figsize = (20, 5))

# FIX: Remove fontdict and use fontweight directly to avoid the alias error
fig.suptitle("Samples from Training Set Batch", fontsize = 16, fontweight = 'bold')

# train_dataset[0][0] accesses the first batch of images
for curr_axis, curr_image in zip(axes.flatten(), train_dataset[0][0][:16]):
    curr_axis.imshow(tf.squeeze(curr_image), cmap = 'gray')
    curr_axis.axis(False)

fig.tight_layout()
plt.show()

### **7.2 Validation Data Images Glimpse**

In [ ]:
fig, axes = plt.subplots(nrows = 2, ncols = 8, figsize = (20, 5))

# FIX: Remove fontdict and use fontweight directly
fig.suptitle("Samples from Validation Set Batch", fontsize = 16, fontweight = 'bold')

# Loop through the first 16 images of the first validation batch
for curr_axis, curr_image in zip(axes.flatten(), validation_dataset[0][0][:16]):
    curr_axis.imshow(tf.squeeze(curr_image), cmap = 'gray')
    curr_axis.axis(False)

fig.tight_layout()
plt.show()

### **7.3 Testing Data Images Glimpse**

In [ ]:
fig, axes = plt.subplots(nrows = 2, ncols = 8, figsize = (20, 5))

# FIX: Remove fontdict and use fontweight directly as a keyword argument
fig.suptitle("Samples from Testing Set Batch", fontsize = 16, fontweight = 'bold')

# Ensure test_dataset is indexed correctly for the first 16 images
for curr_axis, curr_image in zip(axes.flatten(), test_dataset[0][0][:16]):
    curr_axis.imshow(tf.squeeze(curr_image), cmap = 'gray')
    curr_axis.axis(False)

fig.tight_layout()
plt.show()

## **8. Model Development**

In [ ]:
early_stopping = EarlyStopping(monitor = 'val_accuracy', patience = 10)

In [ ]:
ROOT_CHECKPOINT_DIR_PATH = os.path.join(ROOT_DIR, "Model-Checkpoints")
MLP_CHECKPOINT_DIR_PATH = os.path.join(ROOT_CHECKPOINT_DIR_PATH, "Multi-Layer-Perceptron")
ALEXNET_CHECKPOINT_DIR_PATH = os.path.join(ROOT_CHECKPOINT_DIR_PATH, "AlexNet-CNN")
INCEPTIONV3_CHECKPOINT_DIR_PATH = os.path.join(ROOT_CHECKPOINT_DIR_PATH, "InceptionV3")

# Create the directories if they don't exist
os.makedirs(ROOT_CHECKPOINT_DIR_PATH, exist_ok=True)
os.makedirs(MLP_CHECKPOINT_DIR_PATH, exist_ok=True)
os.makedirs(ALEXNET_CHECKPOINT_DIR_PATH, exist_ok=True)
os.makedirs(INCEPTIONV3_CHECKPOINT_DIR_PATH, exist_ok=True)

assert os.path.isdir(ROOT_CHECKPOINT_DIR_PATH) and os.path.isdir(MLP_CHECKPOINT_DIR_PATH) and os.path.isdir(ALEXNET_CHECKPOINT_DIR_PATH) and os.path.isdir(INCEPTIONV3_CHECKPOINT_DIR_PATH)

In [ ]:
mlp_cp_callback = ModelCheckpoint(os.path.join(MLP_CHECKPOINT_DIR_PATH, 'mlp_model_{epoch:02d}-{val_accuracy:.4f}.weights.h5'),
                                  monitor = 'val_accuracy',
                                  verbose = 1,
                                  save_weights_only = True,
                                  save_freq =  'epoch')

alexnet_cp_callback = ModelCheckpoint(os.path.join(ALEXNET_CHECKPOINT_DIR_PATH, 'alexnet_model_{epoch:02d}-{val_accuracy:.4f}.weights.h5'),
                                      monitor = 'val_accuracy',
                                      verbose = 1,
                                      save_weights_only = True,
                                      save_freq = 'epoch')

inceptionv3_cp_callback = ModelCheckpoint(os.path.join(INCEPTIONV3_CHECKPOINT_DIR_PATH, 'inceptionv3_model_{epoch:02d}-{val_accuracy:.4f}.weights.h5'),
                                          monitor = 'val_accuracy',
                                          verbose = 1,
                                          save_weights_only = True,
                                          save_freq = 'epoch')

In [ ]:
def training_process_viz(training_stats: pd.DataFrame, **plot_kwargs) -> None:
    fig, axes = plt.subplots(ncols = 2, figsize = (15, 5))
    fig.suptitle(plot_kwargs['plot_title'], fontsize = 16, fontdict = dict(weight = 'bold'), y = 1.08)
    for curr_axis, col_name in zip(axes, ['accuracy', 'loss']):
        curr_axis.grid(True, alpha = 0.3)
        curr_axis.set_title(f"Model {col_name}".title(), fontsize = 14)
        sns.lineplot(x = range(1, 1 + training_stats.shape[0]), y = training_stats[col_name], color = 'blue', ax = curr_axis)
        sns.lineplot(x = range(1, 1 + training_stats.shape[0]), y = training_stats[f"val_{col_name}"], color = 'red', ax = curr_axis)
        curr_axis.set_xlabel("Epochs", fontsize = 12)
        curr_axis.set_ylabel(col_name.title(), fontsize = 12)
        curr_axis.tick_params(which = 'major', labelsize = 12)
        curr_axis.legend([col_name.title(), f"validation {col_name}".title()], title = col_name.title())
    fig.tight_layout()
    sns.despine()

In [ ]:
def confusion_matrix_viz(model, test_dataset, **plot_kwargs) -> None:
    assert isinstance(model, Sequential)
    model_preds = [np.argmax(curr_row) for curr_row in model.predict(test_dataset)]
    fig, axis = plt.subplots(figsize = (8, 6))
    class_names = ['Glioma', 'Meningioma', 'No-Tumor', 'Pituitary\nTumor']
    sns.heatmap(confusion_matrix(test_dataset.classes, model_preds), annot = True, cmap = plt.cm.Reds, ax = axis)
    axis.set_title(plot_kwargs['plot_title'], fontsize = 14)
    axis.tick_params(which = 'major', labelsize = 12)
    axis.set_xlabel("Pedicted Class", fontsize = 12)
    axis.set_ylabel("Actual Class", fontsize = 12)
    axis.set_xticklabels(class_names, fontdict = dict(fontsize = 12))
    axis.set_yticklabels(class_names, fontdict = dict(fontsize = 12))
    fig.tight_layout()
    sns.despine()

In [ ]:
def generate_report(*models, test_dataset, row_indexes) -> pd.DataFrame:
    assert len(models)
    report_df = pd.DataFrame(columns = ['MAE', 'MSE', 'RMSE', 'Loss', 'Accuracy', 'F1-Score'])
    y_hat = test_dataset.classes # y_hat = ground_truth
    for curr_index, curr_model in enumerate(models):
        assert isinstance(curr_model, Sequential)
        curr_model_loss, curr_model_accuracy = curr_model.evaluate(test_dataset)
        y_preds = [np.argmax(curr_preds) for curr_preds in curr_model.predict(test_dataset)]
        report_df.loc[curr_index] = [mean_absolute_error(y_hat, y_preds), mean_squared_error(y_hat, y_preds), mean_squared_error(y_hat, y_preds, squared = False), curr_model_loss, curr_model_accuracy, f1_score(y_hat, y_preds, average = "micro")]
    report_df.index = row_indexes
    return report_df

### **8.1 Multi-Layer Perceptron**

#### **8.1.1 Development of Multi-Layer Perceptron Model**

In [ ]:
mlp_model = Sequential()
mlp_model.add(Flatten(input_shape = (image_size, image_size, 3), name = 'Flatten-Layer'))
mlp_model.add(Dense(2048, activation = 'relu', name = 'Hidden-Layer-1'))
mlp_model.add(Dropout(rate = 0.2, name = 'Dropout-Layer-1'))
mlp_model.add(Dense(1024, activation = 'relu', name = 'Hidden-Layer-2'))
mlp_model.add(Dropout(rate = 0.2, name = 'Dropout-Layer-2'))
mlp_model.add(Dense(512, activation = 'relu', name = 'Hidden-Layer-3'))
mlp_model.add(Dropout(rate = 0.2, name = 'Dropout-Layer-3'))
mlp_model.add(Dense(4, activation = 'softmax', name = 'Output-Layer-1'))
mlp_model.compile(optimizer = 'Adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])
mlp_model.summary()

#### **8.1.2 Training and Validation of Multi-Layer Perceptron Based Model**

In [ ]:
with tf.device(gpu_device_location) if gpu_device_location else tpu_strategy.scope() if tpu_device_location else tf.device(cpu_device_location):
    mlp_train_history = mlp_model.fit(train_dataset,
                                      batch_size = batch_size,
                                      validation_data = validation_dataset,
                                      epochs = 100,
                                      callbacks = [early_stopping])

#### **8.1.3 Multi-Layer Perceptron Based Model Training Process Statistics**

In [ ]:
training_process_viz(pd.DataFrame(mlp_train_history.history),
                     plot_title = 'Multilayer Perceptron Training Statistics')

#### **8.1.4 Confusion Matrix for Multi-Layer Perceptron Based Model**

In [ ]:
confusion_matrix_viz(mlp_model, train_dataset, plot_title = "MLP Confusion Matrix")

In [ ]:
mlp_report_df = generate_report(mlp_model,
                                test_dataset = test_dataset,
                                row_indexes = ("Multi-Layer-Perceptron Model",))
mlp_report_df

### **8.2 AlexNet CNN**

#### **8.2.1 Develoment of AlexNet CNN Model**

In [ ]:
alexnet_cnn = Sequential()
alexnet_cnn.add(Conv2D(96, kernel_size = 11, strides = 4, activation = 'relu', input_shape = (image_size, image_size, 3), name = 'Conv2D-1'))
alexnet_cnn.add(BatchNormalization(name = 'Batch-Normalization-1'))
alexnet_cnn.add(MaxPool2D(pool_size = 3, strides = 2, name = 'Max-Pooling-1'))
alexnet_cnn.add(Conv2D(256, kernel_size = 5, padding = 'same', activation = 'relu', name = 'Conv2D-2'))
alexnet_cnn.add(BatchNormalization(name = 'Batch-Normalization-2'))
alexnet_cnn.add(MaxPool2D(pool_size = 3, strides = 2, name = 'Max-Pooling-2'))
alexnet_cnn.add(Conv2D(384, kernel_size = 3, padding = 'same', activation = 'relu', name = 'Conv2D-3'))
alexnet_cnn.add(BatchNormalization(name = 'Batch-Normalization-3'))
alexnet_cnn.add(Conv2D(384, kernel_size = 3, padding = 'same', activation = 'relu', name = 'Conv2D-4'))
alexnet_cnn.add(BatchNormalization(name = 'Batch-Normalization-4'))
alexnet_cnn.add(Conv2D(256, kernel_size = 3, padding = 'same', activation = 'relu', name = 'Conv2D-5'))
alexnet_cnn.add(BatchNormalization(name = 'Batch-Normalization-5'))
alexnet_cnn.add(MaxPool2D(pool_size = 3, strides = 2, name = 'Max-Pooling-3'))
alexnet_cnn.add(Flatten(name = 'Flatten-Layer-1'))
alexnet_cnn.add(Dense(128, activation = 'relu', name = 'Hidden-Layer-1'))
alexnet_cnn.add(Dropout(rate = 0.5, name = 'Dropout-Layer-1'))
alexnet_cnn.add(Dense(64, activation = 'relu', name = 'Hidden-Layer-2'))
alexnet_cnn.add(Dropout(rate = 0.5, name = 'Dropout-Layer-2'))
alexnet_cnn.add(Dense(4, activation = 'softmax', name = 'Output-Layer'))
alexnet_cnn.compile(optimizer = 'Adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])
alexnet_cnn.summary()

In [ ]:
alexnet_cnn = Sequential()
alexnet_cnn.add(Conv2D(96, kernel_size = 11, strides = 4, activation = 'relu', input_shape = (image_size, image_size, 3), name = 'Conv2D-1'))
alexnet_cnn.add(BatchNormalization(name = 'Batch-Normalization-1'))
alexnet_cnn.add(MaxPool2D(pool_size = 3, strides = 2, name = 'Max-Pooling-1'))
alexnet_cnn.add(Conv2D(256, kernel_size = 5, padding = 'same', activation = 'relu', name = 'Conv2D-2'))
alexnet_cnn.add(BatchNormalization(name = 'Batch-Normalization-2'))
alexnet_cnn.add(MaxPool2D(pool_size = 3, strides = 2, name = 'Max-Pooling-2'))
alexnet_cnn.add(Conv2D(384, kernel_size = 3, padding = 'same', activation = 'relu', name = 'Conv2D-3'))
alexnet_cnn.add(BatchNormalization(name = 'Batch-Normalization-3'))
alexnet_cnn.add(Conv2D(384, kernel_size = 3, padding = 'same', activation = 'relu', name = 'Conv2D-4'))
alexnet_cnn.add(BatchNormalization(name = 'Batch-Normalization-4'))
alexnet_cnn.add(Conv2D(256, kernel_size = 3, padding = 'same', activation = 'relu', name = 'Conv2D-5'))
alexnet_cnn.add(BatchNormalization(name = 'Batch-Normalization-5'))
alexnet_cnn.add(MaxPool2D(pool_size = 3, strides = 2, name = 'Max-Pooling-3'))
alexnet_cnn.add(Flatten(name = 'Flatten-Layer-1'))
alexnet_cnn.add(Dense(128, activation = 'relu', name = 'Hidden-Layer-1'))
alexnet_cnn.add(Dropout(rate = 0.5, name = 'Dropout-Layer-1'))
alexnet_cnn.add(Dense(64, activation = 'relu', name = 'Hidden-Layer-2'))
alexnet_cnn.add(Dropout(rate = 0.5, name = 'Dropout-Layer-2'))
alexnet_cnn.add(Dense(4, activation = 'softmax', name = 'Output-Layer'))
alexnet_cnn.compile(optimizer = 'Adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])
alexnet_cnn.summary()

#### **8.2.2 Training and Validation of AlexNet CNN Model**

In [ ]:
with tf.device(gpu_device_location) if gpu_device_location else tpu_strategy.scope() if tpu_device_location else tf.device(cpu_device_location):
    alexnet_train_history = alexnet_cnn.fit(train_dataset,
                                            batch_size = batch_size,
                                            validation_data = validation_dataset,
                                            epochs = 100,
                                            callbacks = [early_stopping, alexnet_cp_callback])

#### **8.2.3 AlexNet CNN Model Training Process Statistics**

In [ ]:
training_process_viz(pd.DataFrame(alexnet_train_history.history), plot_title = 'AlexNet CNN Training Stats')

#### **8.2.4 Confusion Matrix for AlexNet CNN Model**

In [ ]:
with tf.device(gpu_device_location) if gpu_device_location else tpu_strategy.scope() if tpu_device_name else tf.device(cpu_device_location):
    confusion_matrix_viz(alexnet_cnn,
                         test_dataset,
                         plot_title = "AlexNet CNN Confusion Matrix")

In [ ]:
alexnet_report_df = generate_report(alexnet_cnn, test_dataset = test_dataset, row_indexes = ['AlexNet CNN'])
alexnet_report_df

### **8.3 Inception V3**

#### **8.3.1 Developement of InceptionV3**

In [ ]:
inception_v3_model = InceptionV3(include_top = False,
                                 input_shape = (image_size, image_size, 3),
                                 pooling = 'avg')
inception_v3_model.trainable = False

In [ ]:
inception_cnn_model = Sequential()
inception_cnn_model.add(inception_v3_model)
inception_cnn_model.add(Flatten())
inception_cnn_model.add(Dense(1024, activation = 'relu', name = 'Hidden-Layer-1'))
inception_cnn_model.add(Dense(4, activation = 'softmax', name = 'Output-Layer'))
inception_cnn_model.compile(optimizer = 'Adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])
inception_cnn_model.summary()

#### **8.3.2 Training and Validation of InceptionV3 Model**

In [ ]:
with tf.device(gpu_device_location) if gpu_device_location else tpu_strategy.scope() if tpu_device_location else tf.device(cpu_device_location):
    inception_model_train_history = inception_cnn_model.fit(train_dataset,
                                                            batch_size = batch_size,
                                                            validation_data = validation_dataset,
                                                            epochs = 100,
                                                            callbacks = [early_stopping, inceptionv3_cp_callback])

#### **8.3.3 InceptionV3 Model Training Process Statistics**

In [ ]:
training_process_viz(pd.DataFrame(inception_model_train_history.history),
                     plot_title = 'Inception-V3 Training Statistics')

#### **8.3.4 Confusion Matrix for InceptionV3 Model**

In [ ]:
with tf.device(gpu_device_location) if gpu_device_location else tpu_strategy.scope() if tpu_device_location else tf.device(cpu_device_name):
    confusion_matrix_viz(inception_cnn_model,
                         test_dataset,
                         plot_title = "Inception-V3 Confusion Matrix")

In [ ]:
inceptionv3_report_df = generate_report(inception_cnn_model, test_dataset = test_dataset, row_indexes = ['InceptionV3'])
inceptionv3_report_df

## **9. Conclusions**

* The **pre-trained (imagenet) InceptionV3** model has performed the best among Multi-Layer perceptron and AlexNet CNN models with an accuracy of $82.57\%$ (Refer the following table).

In [ ]:
final_report_df = pd.concat([mlp_report_df, alexnet_report_df, inceptionv3_report_df])
final_report_df

## **10. Future Works**  

* To incorporate a Data Augmentation pipeline to efficiently generate various different variants of the iamges to make the model more roboust.  

* Training process will be migrated to TPUs (Tensor Processing Units) by representing the data in TFRecord format for significant reduction in training time.  

* Implementation of R-CNN to not only detect a image which has a tumor in it but to also label the location of the tumor in the image.  
